# Aula 04 - Pré-processamento e Feature Engineering

**Módulo 03 IN** - Lógica para predição com inteligência artificial
**19/08/2026 - Sprint 2 - Prof. Ovidio Lopes da Cruz Netto**

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/canaldoovidio/2026-2A-M03/blob/main/notebooks/aula04.ipynb)

## O que este notebook é

É a construção da **base analítica** do case, a tabela que a Aula 05 recebe para treinar o
primeiro modelo. A Aula 03 explorou as cinco séries uma a uma, cada dupla com a sua. Aqui elas
viram uma tabela só, com uma coluna por série, mais as colunas que o modelo precisa e que não
existem no dado cru: defasagem, sazonalidade codificada e escala comparável.

A segunda fonte da junção é o próprio SIDRA. A decisão e o motivo estão em `docs/adrs/ADR-007`.

## Ao final deste notebook você terá

1. unido as cinco séries pela chave `periodo`, nos dois tipos de junção, e visto o que muda;
2. decidido entre `dropna()` e `fillna()` sobre os 40 valores ausentes que o `outer join` produz;
3. criado as features de defasagem de 1 e de 4 trimestres com `shift()`, sabendo o que elas custam;
4. codificado a sazonalidade nas duas formas (dummies e seno/cosseno) e medido o ganho de cada uma;
5. padronizado as cinco colunas numéricas com `StandardScaler`;
6. comparado a correlação em nível com a correlação sobre a primeira diferença;
7. rodado o teste de Shapiro-Wilk das cinco séries, que é o resultado medido pedido pela ART.5.

## 1. Onde estão os arquivos

Mesma resolução de caminho das aulas anteriores: funciona no repositório clonado (onde os CSVs
estão em `../dados/`) e no Colab (onde são baixados da versão publicada do repositório).

In [ ]:
import os
import urllib.request

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

SERIES = [
    "abate_bovinos",
    "abate_suinos",
    "abate_frangos",
    "producao_ovos",
    "producao_leite",
]

BASE_LOCAL = os.path.join("..", "dados")
BASE_BRUTA = ("https://raw.githubusercontent.com/canaldoovidio/2026-2A-M03/"
              "main/dados/")

caminhos = {}
for nome in SERIES:
    arquivo = nome + ".csv"
    local = os.path.join(BASE_LOCAL, arquivo)
    if os.path.exists(local):
        caminhos[nome] = local
    else:
        if not os.path.exists(arquivo):
            urllib.request.urlretrieve(BASE_BRUTA + arquivo, arquivo)
        caminhos[nome] = arquivo

for nome, caminho in caminhos.items():
    print("%-16s -> %s" % (nome, caminho))

## 2. Bloco 1: a junção das cinco séries

A chave é `periodo`, presente nos cinco arquivos, no mesmo formato (`"1997-T1"`).

Duas decisões acompanham a chave. A primeira é o **nome das colunas**: as cinco séries têm a
coluna chamada `valor`, e sem renomear cada uma antes da junção o pandas resolve a colisão
sozinho, criando `valor_x` e `valor_y`. A segunda é o **tipo de junção**, e é ela que a próxima
célula compara.

In [ ]:
def montar(tipo):
    """Une as cinco series pela chave periodo, com o tipo de juncao pedido."""
    base = None
    for nome in SERIES:
        coluna = (pd.read_csv(caminhos[nome])[["periodo", "valor"]]
                  .rename(columns={"valor": nome}))
        base = coluna if base is None else base.merge(coluna, on="periodo", how=tipo)
    return base.sort_values("periodo").reset_index(drop=True)


base_inner = montar("inner")
base_outer = montar("outer")

print("inner:", base_inner.shape, base_inner["periodo"].iloc[0], "ate", base_inner["periodo"].iloc[-1])
print("outer:", base_outer.shape, base_outer["periodo"].iloc[0], "ate", base_outer["periodo"].iloc[-1])

In [ ]:
base_outer.isna().sum()

Os 40 ausentes aparecem em quatro colunas, e nenhum deles é falha de coleta: `producao_ovos`
começa em `1987-T1` e as outras quatro em `1997-T1`. São os dez anos em que só existe ovo, a
mesma diferença de `count` que a Aula 03 mediu.

`inner` responde "quais trimestres têm as cinco medições". `outer` responde "quais trimestres têm
alguma medição". Os dois estão certos; a pergunta do case é a primeira, porque o modelo da Aula 05
recebe as cinco colunas de uma vez.

## 3. Decidir entre `dropna()` e `fillna()`

A Aula 03 terminou com `isna()` devolvendo zero nas cinco séries, e registrou que a escolha entre
diagnosticar e agir ficaria para hoje. A célula abaixo mostra o que cada opção **afirma** sobre o
período sem medição, olhando o primeiro trimestre da base `outer`.

In [ ]:
primeira = base_outer["periodo"].iloc[0]
colunas = ["periodo", "abate_bovinos", "producao_ovos"]

comparacao = pd.DataFrame({
    "periodo": base_outer["periodo"].head(3),
    "cru": base_outer["abate_bovinos"].head(3),
    "fillna(0)": base_outer["abate_bovinos"].fillna(0).head(3),
    "fillna(media)": base_outer["abate_bovinos"].fillna(base_outer["abate_bovinos"].mean()).head(3),
    "bfill": base_outer["abate_bovinos"].bfill().head(3),
})

print("primeiro trimestre da base outer:", primeira)
print()
print(comparacao.to_string(index=False))
print()
print("linhas apos dropna():", len(base_outer.dropna()), "de", len(base_outer))

`fillna(0)` afirmaria que o Brasil abateu zero boi em 1987. `fillna(media)` afirmaria que 1987 se
parecia com a média de 29 anos. `bfill` afirmaria que a produção ficou parada nos dez anos
anteriores ao primeiro registro. Nenhuma das três é defensável aqui, e `dropna()` devolve
exatamente as 117 linhas do `inner`.

Preencher é legítimo quando existe uma afirmação defensável sobre o valor que faltou (sensor com
falha pontual, item não respondido em um questionário). O critério é esse.

**A partir daqui o notebook trabalha sobre `base_inner`.**

In [ ]:
base = base_inner.copy()
base["ano"] = base["periodo"].str[:4].astype(int)
base["trimestre"] = base["periodo"].str[-1].astype(int)
base.head()

## 4. Bloco 2: features de defasagem

O TAPI da LDC proíbe modelo de série temporal, e a ADR-003 registrou a consequência: o case usa
regressão tabular. Um regressor tabular trata cada linha como observação independente e não sabe
que a linha de cima é o trimestre anterior.

`shift(k)` resolve isso descendo a coluna *k* posições, o que traz o passado para dentro da linha
do presente. O topo fica com `NaN`, porque não existe histórico anterior ao início da série.

In [ ]:
ALVO = "abate_frangos"

base["frangos_lag1"] = base[ALVO].shift(1)
base["frangos_lag4"] = base[ALVO].shift(4)

print(base[["periodo", ALVO, "frangos_lag1", "frangos_lag4"]].head(6).to_string(index=False))
print()
print("NaN em lag1:", int(base["frangos_lag1"].isna().sum()))
print("NaN em lag4:", int(base["frangos_lag4"].isna().sum()))
print("linhas apos dropna():", len(base.dropna()), "de", len(base))

`lag1` é a inércia de curto prazo: rebanho, capacidade instalada e contratos não mudam em três
meses. `lag4` é o ciclo anual: compara T3 com T3, sem misturar estação do ano.

A célula abaixo mede as duas defasagens nas cinco séries. Repare em qual delas quebra o padrão.

In [ ]:
print("%-16s %12s %12s   %s" % ("serie", "corr lag1", "corr lag4", "lidera"))
for nome in SERIES:
    l1 = base[nome].corr(base[nome].shift(1))
    l4 = base[nome].corr(base[nome].shift(4))
    lider = "lag1" if l1 > l4 else "lag4"
    print("%-16s %+12.4f %+12.4f   %s" % (nome, l1, l4, lider))

`producao_leite` é a única série em que o mesmo trimestre do ano anterior prediz melhor que o
trimestre imediatamente anterior. É a mesma série que a Aula 03 identificou como a mais sazonal
das cinco: quando a estação do ano domina o comportamento, o vizinho mais parecido é o do ano
passado.

**Vazamento temporal.** O argumento de `shift` é sempre positivo em previsão. `shift(-1)` traz o
valor do próximo trimestre para a linha de hoje, e o modelo treinado assim acerta em validação
(recebeu a resposta como entrada) e falha em produção, onde o próximo trimestre ainda não
aconteceu. Nenhum aviso é emitido: o código roda e a métrica sobe.

## 5. Bloco 3: codificar a sazonalidade

`trimestre` é categórica **nominal**: T1 a T4 são rótulos, e T4 não vale quatro vezes T1. Passar o
número cru como entrada afirmaria uma ordem e uma proporção que não existem.

Duas codificações, e `drop_first=True` não é economia de espaço: com as quatro dummies presentes a
soma delas é sempre 1, uma é combinação linear das outras três, e a matriz perde posto. A categoria
descartada vira a referência dos coeficientes.

In [ ]:
dummies = pd.get_dummies(base["trimestre"], prefix="tri", drop_first=True)
base = pd.concat([base, dummies.astype(float)], axis=1)

base["sen"] = np.sin(2 * np.pi * base["trimestre"] / 4)
base["cos"] = np.cos(2 * np.pi * base["trimestre"] / 4)

print(base[["periodo", "trimestre", "tri_2", "tri_3", "tri_4", "sen", "cos"]]
      .head(5).to_string(index=False))

Qual das duas codificações vale mais? A resposta depende de quanta sazonalidade a série tem para
capturar. A célula abaixo ajusta três regressões por série (só tendência, tendência com dummies,
tendência com seno/cosseno) e compara o R².

In [ ]:
def r2(X, y):
    return LinearRegression().fit(X, y).score(X, y)


tempo = np.arange(len(base)).reshape(-1, 1)
D = base[["tri_2", "tri_3", "tri_4"]].to_numpy()
S = base[["sen", "cos"]].to_numpy()

print("%-16s %12s %12s %12s   %s" % ("serie", "tendencia", "+ dummies", "+ sen/cos", "amplitude"))
for nome in SERIES:
    y = base[nome].to_numpy()
    grupo = base.groupby("trimestre")[nome].mean()
    amplitude = (grupo.max() - grupo.min()) / base[nome].mean() * 100
    print("%-16s %12.4f %12.4f %12.4f   %6.2f pp"
          % (nome, r2(tempo, y), r2(np.hstack([tempo, D]), y),
             r2(np.hstack([tempo, S]), y), amplitude))

Com quatro categorias, as duas codificações entregam praticamente o mesmo, e seno/cosseno faz isso
com uma coluna a menos. A vantagem cresce com o tamanho do ciclo: base mensal são 11 dummies contra
2 colunas; semanal, 51 contra 2. Em compensação, dummies capturam qualquer padrão entre as
categorias, e o par seno/cosseno captura apenas o ciclo suave.

A amplitude da última coluna explica por que o ganho varia tanto entre as séries: feature que não
tem o que capturar não melhora modelo.

## 6. Escalonamento das variáveis numéricas

As cinco colunas medem coisas diferentes, em unidades diferentes, com quatro ordens de grandeza
entre a maior e a menor. Ao colocá-las na mesma matriz de entrada, a coluna de frango domina
qualquer cálculo de distância só por ser numericamente maior.

In [ ]:
print(base[SERIES].agg(["mean", "std"]).T.to_string())
print()

escalador = StandardScaler()
Z = escalador.fit_transform(base[SERIES])

print("apos StandardScaler:")
print("  media por coluna:", np.round(Z.mean(axis=0), 12))
print("  desvio por coluna:", np.round(Z.std(axis=0), 12))

O resultado é adimensional: 1,5 significa "um desvio e meio acima da média desta série", e isso é
comparável entre frango e ovo, o que os números crus não são.

Regressão linear sem regularização produz as mesmas previsões com ou sem padronização. Precisam de
escala comparável: KNN e SVM (medem distância), regressão regularizada (a penalidade depende da
unidade do coeficiente) e PCA. Três deles aparecem entre as Aulas 05 e 08.

**A ordem importa.** Média e desvio precisam ser calculados só sobre o treino e depois aplicados ao
teste. Padronizar a base inteira antes de dividir faz a média do teste entrar no cálculo. A Aula 05
trata isso com `fit` no treino e `transform` nos dois.

## 7. Seleção de característica: a armadilha da tendência

Com a base montada, qual coluna vale a pena manter? O primeiro instinto é ranquear por correlação
com o alvo. A célula abaixo faz isso duas vezes: sobre o nível e sobre a primeira diferença.

In [ ]:
diferencas = base[SERIES].diff().dropna()

print("%-16s %14s %14s" % ("coluna", "corr nivel", "corr diff"))
for nome in SERIES:
    if nome == ALVO:
        continue
    print("%-16s %+14.4f %+14.4f"
          % (nome, base[ALVO].corr(base[nome]), diferencas[ALVO].corr(diferencas[nome])))

As duas medidas saem das mesmas linhas, com o mesmo código; a única diferença é o `.diff()`
aplicado antes. Em nível, as quatro colunas parecem excelentes preditoras. Sobre a variação
trimestre a trimestre, três ficam abaixo de 0,21 e leite fica praticamente em zero.

As cinco séries crescem ao longo de 29 anos, e duas séries crescentes correlacionam alto entre si
mesmo sem relação nenhuma, porque a correlação mede a tendência de longo prazo nas duas.
Ranquear característica por correlação em nível, nesta base, selecionaria a tendência cinco vezes
seguidas e chamaria isso de cinco preditoras.

Isso não descarta as quatro colunas. Significa que o critério precisa ser aplicado sobre a série
transformada, e que a decisão final se toma com desempenho preditivo em dados que o modelo não viu,
o que a Aula 05 torna possível ao separar treino e teste por corte temporal.

## 8. Normalidade: o teste de hipótese da ART.5

A ART.5 pede um teste conduzido, com estatística e valor-p. Shapiro-Wilk tem como hipótese nula
que a amostra vem de uma distribuição normal, e um valor-p baixo rejeita essa hipótese.

In [ ]:
print("%-16s %10s %12s %12s   %s" % ("serie", "W", "p (nivel)", "p (diff)", "conclusao na diff"))
for nome in SERIES:
    W, p = stats.shapiro(base[nome])
    Wd, pd_ = stats.shapiro(base[nome].diff().dropna())
    conclusao = "rejeita" if pd_ < 0.05 else "nao rejeita a normalidade"
    print("%-16s %10.4f %12.6f %12.5f   %s" % (nome, W, p, pd_, conclusao))

As cinco séries rejeitam normalidade em nível, e o motivo é estrutural: uma série com tendência de
crescimento não tem como ser normal, porque a média muda ao longo da amostra. Depois de `.diff()`,
`abate_bovinos` deixa de rejeitar e as outras quatro continuam rejeitando, uma delas por pouco.

**Rejeitar normalidade não invalida a regressão.** Mínimos quadrados não exige entradas normais. A
suposição de normalidade em regressão linear é sobre os **resíduos** do modelo ajustado, e serve
para os testes de significância dos coeficientes. O teste de hoje é sobre as variáveis, e responde
à ART.5; o teste sobre resíduos vem depois de existir um modelo, na Aula 05.

## 9. A base analítica pronta

É esta tabela que a Aula 05 recebe. Repare que ela é reconstruída inteira a cada execução, a partir
dos CSVs crus: nada aqui é gravado em disco, e é isso que permite mudar uma decisão de codificação
e refazer tudo sem consultar o IBGE de novo.

In [ ]:
COLUNAS_MODELO = (["periodo", "ano", "trimestre"] + SERIES
                  + ["frangos_lag1", "frangos_lag4", "tri_2", "tri_3", "tri_4", "sen", "cos"])

analitica = base[COLUNAS_MODELO].dropna().reset_index(drop=True)

print("base analitica:", analitica.shape)
print("de", analitica["periodo"].iloc[0], "ate", analitica["periodo"].iloc[-1])
print()
print(analitica[["periodo", ALVO, "frangos_lag1", "frangos_lag4", "sen", "cos"]]
      .head(4).to_string(index=False))

## 10. Desafio

Responda no código, e o item 4 em texto.

1. Monte a base analítica da série **da sua dupla** (não do frango): junção `inner`, defasagens de
   1 e de 4 trimestres, dummies de trimestre e as cinco colunas padronizadas.
2. Diga qual das duas defasagens correlaciona mais com o valor corrente na sua série, com quatro
   casas decimais.
3. Rode Shapiro-Wilk na sua série, em nível e sobre a primeira diferença, e reporte W e o valor-p
   dos dois.
4. Em `resposta_4`, escreva em uma frase o que a comparação entre a correlação em nível e a
   correlação sobre a primeira diferença revelou sobre a sua série, e o que isso muda na escolha
   das colunas que entram no modelo da Aula 05.

In [ ]:
MINHA_SERIE = "producao_leite"   # troque pela serie da sua dupla

# 1. base analitica da serie escolhida
minha = base[["periodo", "trimestre", MINHA_SERIE]].copy()
minha["lag1"] = minha[MINHA_SERIE].shift(1)
minha["lag4"] = minha[MINHA_SERIE].shift(4)
minha = pd.concat([minha, pd.get_dummies(minha["trimestre"], prefix="tri",
                                         drop_first=True).astype(float)], axis=1)
minha = minha.dropna().reset_index(drop=True)
print("base da dupla:", minha.shape)

# 2. qual defasagem correlaciona mais
c1 = minha[MINHA_SERIE].corr(minha["lag1"])
c4 = minha[MINHA_SERIE].corr(minha["lag4"])
print("corr lag1 = %+.4f   corr lag4 = %+.4f   lidera: %s"
      % (c1, c4, "lag1" if c1 > c4 else "lag4"))

# 3. Shapiro-Wilk em nivel e sobre a primeira diferenca
W, p = stats.shapiro(base[MINHA_SERIE])
Wd, pd_ = stats.shapiro(base[MINHA_SERIE].diff().dropna())
print("nivel:    W=%.4f  p=%.6f" % (W, p))
print("diferenca: W=%.4f  p=%.6f" % (Wd, pd_))

# 4. resposta em texto
resposta_4 = (
    "A correlacao de producao_leite com abate_frangos e +0,96 em nivel e -0,04 em "
    "primeira diferenca. O valor alto mede a tendencia comum das duas series ao longo "
    "de 29 anos. Na Aula 05, incluir as outras series como entrada precisa ser "
    "justificado por desempenho preditivo medido em dados que o modelo nao viu."
)
print()
print(resposta_4)